In [ ]:
import os
from typing import List
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

def run_group_comparison_analysis(csv_path: str, output_dir: str = "outputs/plots"):
    """
    Executes a comprehensive statistical analysis pipeline comparing 
    resonant vs. non-resonant singing feature distributions.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load data and clean missing observation artifacts
    # Ensure string 'NaN' values generated by the pipeline are parsed as true numerical NaNs
    df = pd.read_csv(csv_path)
    
    # Identify all columns containing calculated acoustic features
    feature_cols = [col for col in df.columns if col.startswith("feature_")]
    
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    print(f"📊 Loaded dataset: {df.shape[0]} rows across {len(feature_cols)} target features.")
    
    statistical_summary = []
    
    # 2. Iterate through each feature to perform distribution audits and hypothesis tests
    for feature in feature_cols:
        # Separate data cohorts into target condition brackets
        resonant_group = df[df["condition"] == "resonant"][feature].dropna().to_numpy()
        non_resonant_group = df[df["condition"] == "non-resonant"][feature].dropna().to_numpy()
        
        # Guardrail check: Ensure adequate statistical power exists for comparison
        if len(resonant_group) < 3 or len(non_resonant_group) < 3:
            print(f"⚠️ Skipping {feature}: Insufficient sample size.")
            continue
            
        # 3. Normality Distribution Testing (Shapiro-Wilk)
        # H0: The data is normally distributed.
        _, p_shapiro_res = stats.shapiro(resonant_group)
        _, p_shapiro_non = stats.shapiro(non_resonant_group)
        
        # Data is deemed normal if both groups fail to reject the null hypothesis (p > 0.05)
        is_normal = (p_shapiro_res > 0.05) and (p_shapiro_non > 0.05)
        
        # 4. Inferential Testing Selection (Assuming Paired/Within-Subject Design here)
        # Note: If trials or lengths don't match exactly per singer, fall back to Independent testing.
        if len(resonant_group) == len(non_resonant_group):
            test_type = "Paired (Within-Subject)"
            if is_normal:
                stat_val, p_val = stats.ttest_rel(resonant_group, non_resonant_group)
                test_name = "Paired t-test"
            else:
                stat_val, p_val = stats.wilcoxon(resonant_group, non_resonant_group)
                test_name = "Wilcoxon Signed-Rank"
        else:
            test_type = "Independent (Between-Subject)"
            if is_normal:
                # Run Levene's test to check for equality of variances
                _, p_levene = stats.levene(resonant_group, non_resonant_group)
                stat_val, p_val = stats.ttest_ind(resonant_group, non_resonant_group, equal_var=(p_levene > 0.05))
                test_name = "Independent t-test"
            else:
                stat_val, p_val = stats.mannwhitneyu(resonant_group, non_resonant_group, alternative='two-sided')
                test_name = "Mann-Whitney U"
                
        # 5. Calculate Basic Effect Size (Cohen's d for descriptive context)
        # mean_diff = np.mean(resonant_group) - np.mean(non_resonant_group)
        # pooled_std = np.sqrt((np.var(resonant_group, ddof=1) + np.var(non_resonant_group, ddof=1)) / 2.0)
        # cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0.0
        deltas = non_resonant_group - resonant_group
        
        # Append outputs to compilation listing
        statistical_summary.append({
            "Feature": feature,
            "Design_Type": test_type,
            "Normality_Passed": is_normal,
            "Applied_Test": test_name,
            "Statistic": round(float(stat_val), 4),
            "p_value": round(float(p_val), 5),
            "Significance": "* Significant" if p_val < 0.05 else "n.s. (Not Significant)",
            "Cohen_d": round(float(cohens_d), 4)
        })
        
        # 6. Visualization Rendering Layer
        # Mandated rule compliance: No plt.figure() allocations, direct axis saving
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.boxplot(
            data=df, 
            x="condition", 
            y=feature, 
            order=["non-resonant", "resonant"],
            hue="condition", 
            palette="Set2",
            legend=False,
            ax=ax
        )
        sns.stripplot(
            data=df, 
            x="condition", 
            y=feature, 
            order=["non-resonant", "resonant"], 
            hue="condition",
            palette="dark:black", 
            alpha=0.4, 
            size=5, 
            ax=ax
        )
        
        ax.set_title(f"Distribution Analysis: {feature}\n({test_name} p={p_val:.5f})", fontsize=10)
        ax.set_xlabel("Vocal Condition")
        ax.set_ylabel("Extracted Value Metric")
        
        # Save plot immediately using the specific string filename pattern
        plot_filename = os.path.join(output_dir, f"stat_comparison_{feature}.png")
        plt.tight_layout()
        plt.savefig(plot_filename, dpi=200)
        plt.close(fig)

    # 7. Export summary matrix report to disk
    summary_df = pd.DataFrame(statistical_summary)
    summary_df.to_csv("outputs/statistical_inference_report.csv", index=False)
    print("🚀 Statistical exploration pipeline complete. Outputs compiled inside 'outputs/' folder.")

if __name__ == "__main__":
    # Point directly to your master compilation feature spreadsheet file
    run_group_comparison_analysis("outputs/singing_features_master.csv")